# Gate A: retrieval over real transcripts

Spec 07, "Gate A notebook retrieval test". The logic is in `gate_a.py` (tested by `test_gate_a.py`); this notebook runs it.

Before running:

1. Export the passages: `wt export transcripts --format jsonl --out data/export/` against the real-data database.
2. Check the questions in `eval/gate_a_questions.toml`.

Run from the repository root's `scripts/notebooks` directory:
`uv run --project scripts/notebooks jupyter lab scripts/notebooks/gate_a_retrieval.ipynb`.

Everything this writes goes under `data/gate-a/`, which git ignores, because results quote real transcript text.

In [ ]:
import json, os, sys, time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "Cargo.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts" / "notebooks"))
import gate_a as g

EXPORT = ROOT / "data" / "export" / "passages.jsonl"
QUESTIONS = ROOT / "eval" / "gate_a_questions.toml"
MODEL_DIR = Path(os.environ.get("WT_MODEL_DIR", ROOT / "models")) / os.environ.get("WT_EMBED_MODEL", "bge-m3-int8")
OUT = ROOT / "data" / "gate-a"

## Inputs

In [ ]:
passages = g.load_passages(EXPORT)
questions = g.load_questions(QUESTIONS)
teachers = Counter(p.teacher_id for p in passages)
print(f"{len(passages)} passages, {len({p.talk_id for p in passages})} talks, {len(teachers)} teachers, {len(questions)} questions")
print("passages per teacher, largest five:", [n for _, n in teachers.most_common(5)])

## 1. Dense index

bge-m3 int8, the graph the Rust service uses, checked against its `export.json`. The vectors are cached under `data/gate-a/` by passage content and model, so a rerun on the same export skips this step. On a CPU the first run takes a while.

In [ ]:
bge = g.Bge(MODEL_DIR)
t0 = time.time()
def progress(done, total):
    if done % 2000 == 0 or done == total:
        print(f"{done}/{total} embedded, {time.time() - t0:.0f}s", flush=True)
matrix = g.cached_embeddings(passages, OUT / "cache", bge.model_sha, bge.embed, progress)
print(matrix.shape)

## 2. BM25 index

In [ ]:
sparse = g.Sparse(passages)

## 3–4. Fusion and reranking

Reciprocal rank fusion at k = 60 over the top 100 of each, bge-reranker-v2-m3 on the top 40, then the top 10 with at most 3 per teacher.

In [ ]:
reranker = g.Reranker()
retriever = g.Retriever(passages, matrix, sparse, bge.embed, reranker.score)
results = {}
for q in questions:
    results[q.id] = retriever.search(q.text)
    print(q.id, len(results[q.id]), "passages,", len({h.passage.teacher_id for h in results[q.id]}), "teachers")

## 5. Review file

With `DATABASE_URL` set, each passage names its teacher and talk and links to the source. `python scripts/notebooks/gate_a.py data/gate-a/<run>` rebuilds a saved run's page the same way.

In [ ]:
run_id = time.strftime("%Y%m%d-%H%M") + "-" + g.passages_digest(passages)[:8]
run_dir = OUT / run_id
run_dir.mkdir(parents=True, exist_ok=True)
meta = {
    "run_id": run_id,
    "export": str(EXPORT.relative_to(ROOT)),
    "passages": len(passages),
    "passages_digest": g.passages_digest(passages),
    "embed_model": bge.model_id, "embed_model_sha256": bge.model_sha,
    "reranker": f"{g.RERANKER_REPO}@{g.RERANKER_REVISION}",
    "rrf_k": g.RRF_K, "fuse_depth": g.FUSE_DEPTH, "rerank_top": g.RERANK_TOP,
    "keep_top": g.KEEP_TOP, "per_teacher_cap": g.PER_TEACHER_CAP,
    "questions_file": str(QUESTIONS.relative_to(ROOT)),
}
(run_dir / "results.json").write_text(json.dumps(g.results_json(questions, results, meta), indent=2), encoding="utf-8")
# The reviewer sees teacher, talk and source link (decision of October 9, 2026). Names come
# from the database for the returned talks only; without DATABASE_URL the page is blind.
dsn = os.environ.get("DATABASE_URL")
talks = sorted({h.passage.talk_id for hs in results.values() for h in hs})
sources = g.fetch_sources(dsn, talks) if dsn else None
(run_dir / "review.html").write_text(g.review_html(questions, results, g.neighbors(passages), run_id, sources), encoding="utf-8")
print(run_dir / "review.html")

## Scoring

When the reviewer sends back `ratings-<run>.json`, put it in the run's directory and run this cell. Paste the printed block into `docs/gate-a.md`.

In [ ]:
RUN = run_id  # or the directory name of an earlier run
run_dir = OUT / RUN
ratings_file = run_dir / f"ratings-{RUN}.json"
if ratings_file.exists():
    s = g.score(json.loads(ratings_file.read_text(encoding="utf-8")),
                json.loads((run_dir / "results.json").read_text(encoding="utf-8")))
    print(g.gate_record(s))
else:
    print("no ratings yet:", ratings_file)